# Canal Moins — Churn Prediction Model
## Exploratory Data Analysis & Model Development
**Block 4 — MLOps & Production | JHEDA Master Thesis**

> *"We know you'll cancel — we just won't let you."*

---
### Objective
Build a binary classifier that predicts whether a Canal Moins subscriber will **churn in the next 30 days**,
based on their viewing behaviour from the Gold layer (Block 3).

### Contents
1. Data generation & exploration
2. Feature engineering
3. Exploratory Data Analysis (EDA)
4. Model training (XGBoost)
5. Model evaluation & feature importance
6. Business impact analysis

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
import xgboost as xgb
import warnings
warnings.filterwarnings('ignore')

from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    roc_auc_score, f1_score, precision_score,
    recall_score, confusion_matrix, roc_curve
)

plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams['figure.figsize'] = (12, 5)
plt.rcParams['font.size'] = 11

RANDOM_STATE = 42
GREEN = '#00C97B'
RED = '#FF4757'
DARK = '#0A0A0A'

print('✅ Libraries loaded')

## 1. Data Generation
Simulates the `gold_churn_features` table from Block 3.
In production, this would query Redshift directly.

In [ ]:
np.random.seed(RANDOM_STATE)
N = 10000
churn = np.random.choice([0, 1], N, p=[0.75, 0.25])

df = pd.DataFrame({
    'subscriber_id': [f'sub_{i:08d}' for i in range(N)],
    'total_sessions_30d': np.where(churn, np.random.poisson(3, N), np.random.poisson(18, N)).clip(0, 100),
    'total_watch_time_h': np.where(churn, np.abs(np.random.normal(2, 3, N)), np.abs(np.random.normal(25, 15, N))).round(2),
    'avg_completion_rate': np.where(churn, np.random.beta(2, 5, N), np.random.beta(5, 2, N)).round(4),
    'avg_engagement_score': np.where(churn, np.random.normal(25, 15, N), np.random.normal(72, 18, N)).clip(0, 100).round(2),
    'days_since_last_watch': np.where(churn, np.random.exponential(15, N), np.random.exponential(3, N)).clip(0, 30).round(0).astype(int),
    'sessions_last_7d': np.where(churn, np.random.poisson(0.5, N), np.random.poisson(5, N)).clip(0, 30),
    'unique_content_count': np.where(churn, np.random.poisson(2, N), np.random.poisson(12, N)).clip(0, 50),
    'plan_type': np.random.choice(['basic', 'plus', 'elite'], N, p=[0.5, 0.35, 0.15]),
    'country': np.random.choice(['FR', 'BE', 'CH', 'LU'], N, p=[0.7, 0.15, 0.1, 0.05]),
    'favourite_device': np.random.choice(['smart_tv', 'mobile', 'web', 'tablet'], N, p=[0.4, 0.3, 0.2, 0.1]),
    'churned_30d': churn
})

print(f'Dataset: {df.shape[0]:,} rows × {df.shape[1]} columns')
print(f'Churn rate: {df.churned_30d.mean():.1%}')
df.head()

## 2. Exploratory Data Analysis

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Churn distribution
counts = df.churned_30d.value_counts()
axes[0].bar(['Retained', 'Churned'], counts.values, color=[GREEN, RED], alpha=0.85)
axes[0].set_title('Churn Distribution', fontweight='bold')
axes[0].set_ylabel('Subscribers')
for i, v in enumerate(counts.values):
    axes[0].text(i, v + 50, f'{v:,}\n({v/N:.1%})', ha='center', fontsize=10)

# Sessions by churn
df.groupby('churned_30d')['total_sessions_30d'].plot.hist(
    bins=30, alpha=0.7, ax=axes[1], color=[GREEN, RED]
)
axes[1].set_title('Sessions last 30 days — Churned vs Retained', fontweight='bold')
axes[1].set_xlabel('Sessions')
axes[1].legend(['Retained', 'Churned'])

plt.tight_layout()
plt.show()

In [ ]:
# Key stats by churn status
stats = df.groupby('churned_30d')[[
    'total_sessions_30d', 'total_watch_time_h',
    'avg_completion_rate', 'days_since_last_watch'
]].mean().round(2)
stats.index = ['Retained', 'Churned']
print('Average behaviour by churn status:')
stats

## 3. Feature Engineering

In [ ]:
PLAN_ENC = {'basic': 0, 'plus': 1, 'elite': 2}
COUNTRY_ENC = {'BE': 0, 'CH': 1, 'FR': 2, 'LU': 3}
DEVICE_ENC = {'mobile': 0, 'smart_tv': 1, 'tablet': 2, 'web': 3}

df['plan_type_enc'] = df['plan_type'].map(PLAN_ENC)
df['country_enc'] = df['country'].map(COUNTRY_ENC)
df['device_enc'] = df['favourite_device'].map(DEVICE_ENC)
df['sessions_per_day'] = (df['total_sessions_30d'] / 30).round(4)
df['watch_per_session_h'] = np.where(
    df['total_sessions_30d'] > 0,
    df['total_watch_time_h'] / df['total_sessions_30d'], 0
).round(4)
df['recency_x_frequency'] = (
    1 / (df['days_since_last_watch'] + 1) * df['total_sessions_30d']
).round(4)
df['is_elite'] = (df['plan_type'] == 'elite').astype(int)
df['is_mobile_first'] = (df['favourite_device'] == 'mobile').astype(int)

FEATURE_COLS = [
    'total_sessions_30d', 'total_watch_time_h', 'avg_completion_rate',
    'avg_engagement_score', 'days_since_last_watch', 'sessions_last_7d',
    'unique_content_count', 'plan_type_enc', 'country_enc', 'device_enc',
    'sessions_per_day', 'watch_per_session_h', 'recency_x_frequency',
    'is_elite', 'is_mobile_first'
]
print(f'✅ {len(FEATURE_COLS)} features engineered')

## 4. Model Training — XGBoost

In [ ]:
X = df[FEATURE_COLS]
y = df['churned_30d']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=RANDOM_STATE, stratify=y
)

model = xgb.XGBClassifier(
    n_estimators=200, max_depth=6, learning_rate=0.05,
    subsample=0.8, colsample_bytree=0.8,
    scale_pos_weight=3, random_state=RANDOM_STATE,
    eval_metric='auc', early_stopping_rounds=20,
    verbosity=0
)
model.fit(X_train, y_train, eval_set=[(X_test, y_test)], verbose=False)
print('✅ Model trained')

## 5. Evaluation

In [ ]:
y_pred = model.predict(X_test)
y_prob = model.predict_proba(X_test)[:, 1]

auc = roc_auc_score(y_test, y_prob)
f1 = f1_score(y_test, y_pred)
precision = precision_score(y_test, y_pred)
recall = recall_score(y_test, y_pred)

print('Model Performance:')
print(f'  AUC-ROC   : {auc:.4f} {"✅" if auc >= 0.80 else "❌"}')
print(f'  F1 Score  : {f1:.4f}')
print(f'  Precision : {precision:.4f}')
print(f'  Recall    : {recall:.4f}')

# ROC Curve
fpr, tpr, _ = roc_curve(y_test, y_prob)
plt.figure(figsize=(7, 5))
plt.plot(fpr, tpr, color=GREEN, lw=2, label=f'AUC = {auc:.4f}')
plt.plot([0, 1], [0, 1], 'k--', lw=1)
plt.fill_between(fpr, tpr, alpha=0.1, color=GREEN)
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('ROC Curve — Canal Moins Churn Model', fontweight='bold')
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
# Feature importance
importance = pd.Series(
    model.feature_importances_, index=FEATURE_COLS
).sort_values(ascending=True).tail(10)

plt.figure(figsize=(10, 6))
bars = plt.barh(importance.index, importance.values, color=GREEN, alpha=0.85)
plt.xlabel('Feature Importance')
plt.title('Top 10 Most Predictive Features', fontweight='bold')
for bar, val in zip(bars, importance.values):
    plt.text(val + 0.001, bar.get_y() + bar.get_height()/2,
             f'{val:.3f}', va='center', fontsize=9)
plt.tight_layout()
plt.show()

## 6. Business Impact

In [ ]:
SUBSCRIBERS = 8_000_000
MONTHLY_REVENUE = 12
CHURN_RATE = 0.25
RETENTION_RATE = 0.30

churners_detected = int(SUBSCRIBERS * CHURN_RATE * recall)
retained = int(churners_detected * RETENTION_RATE)
revenue_saved = retained * MONTHLY_REVENUE * 12

print('Business Impact Analysis')
print('=' * 40)
print(f'Total subscribers          : {SUBSCRIBERS:>12,}')
print(f'Expected churners/year     : {int(SUBSCRIBERS * CHURN_RATE):>12,}')
print(f'Churners detected by model : {churners_detected:>12,}')
print(f'Subscribers retained (30%) : {retained:>12,}')
print(f'Annual revenue saved       : €{revenue_saved:>11,}')
print('=' * 40)